# 01 — Tyk2 Structure-Based Analysis: Target & SAR

**What this project is:** a structure-based drug design *reasoning* exercise on Tyk2 — prep a real co-crystal, understand the binding site, pull the measured SAR, predict activity rank-order from structure, and analyze where structure-based prediction succeeds and fails. The deliverable is judgment, not a pipeline.

**It pairs with the FEP project (AlchemyBench):** both use the same Tyk2 congeneric series (the Schrödinger JACS benchmark ligands, ejm_/jmc_), so the SBDD reasoning and the relative-binding-FEP results converge on one integrated Tyk2 story.

---
### Critical target detail: JH1 vs JH2

Tyk2 has **two** druggable sites:
- **JH1** — the catalytically active kinase domain, classic ATP-competitive site (PDB e.g. **4GVJ**). *The JACS benchmark ligands bind here.*
- **JH2** — the pseudokinase regulatory domain, where the modern *selective* inhibitors bind (deucravacitinib and the current clinical wave; PDB e.g. 6NZP, 8S98).

**We target JH1** to match the benchmark ligands. Noting that Tyk2's selectivity story actually lives in JH2 is a target-biology point worth making in the writeup — it shows you understand *why* the field moved to the allosteric site, not just how to dock.

## 1. Pull the Tyk2 JH1 co-crystal from the PDB

We fetch 4GVJ (Tyk2 JH1 kinase domain). RCSB serves structures directly; no login needed. For a real project you would then prep this consciously — this notebook documents those choices rather than clicking defaults.

In [ ]:
import os, requests
os.makedirs('../data', exist_ok=True)

PDB_ID = '4GVJ'  # Tyk2 JH1 kinase domain, ATP site
pdb_path = f'../data/{PDB_ID}.pdb'
if not os.path.exists(pdb_path):
    r = requests.get(f'https://files.rcsb.org/download/{PDB_ID}.pdb', timeout=30)
    r.raise_for_status()
    with open(pdb_path, 'w') as f:
        f.write(r.text)
print('Saved', pdb_path, os.path.getsize(pdb_path), 'bytes')

In [ ]:
# Quick look at what's in the structure: chains, resolution, bound ligands (HETATM residues).
hetero = {}
resolution = None
with open(pdb_path) as f:
    for line in f:
        if line.startswith('REMARK   2 RESOLUTION'):
            resolution = line.strip()
        if line.startswith('HETATM'):
            resname = line[17:20].strip()
            hetero[resname] = hetero.get(resname, 0) + 1
print(resolution)
print('Hetero groups (ligand/ion/solvent):')
for k, v in sorted(hetero.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v} atoms')
# The bound inhibitor is usually the largest non-water HETATM group; HOH is water.

### Structure-prep notes (fill in — this is the judgment part)

Document the choices a real modeler makes and *why*. Prompts:

- **Resolution** — is it good enough (< ~2.5 Å) to trust side-chain positions in the pocket?
- **Protonation** — the hinge and catalytic-lysine region: which residues' protonation states matter for ligand H-bonding at pH 7.4?
- **The bound ligand** — what is it, and does it define the ATP pocket you'll reason about?
- **Missing atoms/loops** — any gaps near the binding site? (Grep for REMARK 465 missing residues.)
- **Waters** — any conserved pocket waters worth keeping?

You don't need to *run* a prep tool here; the credible thing is stating the decisions.

## 2. Understand the ATP binding site

Write the pharmacophore of the Tyk2 JH1 ATP pocket in chemical language. The key features of a kinase ATP site, which you should identify in 4GVJ:

- **Hinge region** — backbone H-bond donor/acceptor that nearly all ATP-competitive inhibitors hydrogen-bond to (mimicking adenine). Identify the hinge residue(s).
- **Gatekeeper residue** — controls access to the back hydrophobic pocket; its size drives selectivity.
- **Catalytic Lys / αC-helix Glu** — the conserved salt bridge.
- **DFG motif** — in/out conformation determines type-I vs type-II binding.
- **Glycine-rich (P-)loop** — caps the phosphate region.

*This section is prose you write from the structure + literature. It is the difference between 'I docked something' and 'I understand this target.'*

## 3. Pull the measured SAR from ChEMBL

Query ChEMBL for Tyk2 inhibitors with measured activity. This is the ground-truth SAR you will reason against. (teachopencadd ships the ChEMBL client; this reaches EBI over the network.)

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

# Find the Tyk2 target. UniProt P29597 is human TYK2.
target_client = new_client.target
targets = target_client.filter(target_components__accession='P29597').only(
    ['target_chembl_id', 'pref_name', 'organism', 'target_type'])
targets_df = pd.DataFrame(targets)
print(targets_df)
# Pick the single-protein human TYK2 target id from the output.
TARGET_ID = 'CHEMBL3553'  # human TYK2 single protein — verify against the print above

In [ ]:
# Pull IC50 activities for that target.
activity_client = new_client.activity
acts = activity_client.filter(
    target_chembl_id=TARGET_ID,
    standard_type='IC50',
    relation='=',
).only(['molecule_chembl_id', 'canonical_smiles', 'standard_value',
        'standard_units', 'standard_relation', 'assay_chembl_id'])

acts_df = pd.DataFrame(acts)
print('Raw activities pulled:', len(acts_df))
acts_df.head()

In [ ]:
# Clean: keep nM IC50s with SMILES, convert to pIC50, drop dupes.
import numpy as np

df = acts_df.dropna(subset=['standard_value', 'canonical_smiles']).copy()
df = df[df['standard_units'] == 'nM']
df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
df = df[(df['standard_value'] > 0) & (df['standard_value'] < 1e7)]
df['pIC50'] = -np.log10(df['standard_value'] * 1e-9)  # nM -> M -> pIC50
# Median-aggregate repeated measurements of the same molecule.
df = df.groupby('molecule_chembl_id').agg(
    canonical_smiles=('canonical_smiles', 'first'),
    pIC50=('pIC50', 'median'),
    n_measurements=('pIC50', 'size'),
).reset_index()
print('Unique molecules with clean pIC50:', len(df))
df.sort_values('pIC50', ascending=False).head(10)

In [ ]:
# Save the cleaned SAR set for the analysis notebook.
df.to_csv('../data/tyk2_chembl_sar.csv', index=False)
print('Saved ../data/tyk2_chembl_sar.csv with', len(df), 'molecules')
print('pIC50 range:', round(df.pIC50.min(), 2), '-', round(df.pIC50.max(), 2))

## 4. Locate the benchmark congeneric series within the SAR

Your FEP ligands (ejm_/jmc_) are a specific congeneric series. Here we look at the broader ChEMBL SAR to place them in context: what scaffolds are potent, how wide is the activity range, and where the easy/hard prediction cases sit. This connects the two projects.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem
from rdkit.Chem import PandasTools

# Add RDKit molecules; show the most potent handful.
df_top = df.sort_values('pIC50', ascending=False).head(6).copy()
mols = [Chem.MolFromSmiles(s) for s in df_top['canonical_smiles']]
legends = [f"{r.molecule_chembl_id}  pIC50={r.pIC50:.1f}"
           for _, r in df_top.iterrows()]
Draw.MolsToGridImage([m for m in mols if m], legends=legends, molsPerRow=3,
                     subImgSize=(300, 250))

### SAR reasoning (fill in — the core deliverable)

Now do the analysis that shows judgment:

1. **Scaffold families** — what core(s) dominate the potent end? How do they engage the hinge you identified in section 2?
2. **Activity range** — how many log units does the set span? A wide range means rank-order prediction is feasible; a narrow one means noise dominates (and docking will struggle).
3. **Predict before you check** — pick ~5 congeneric molecules, and from structural reasoning *predict* their potency order. Record the prediction here.
4. **Where structure-based prediction will fail** — which pairs are 'activity cliffs' (tiny structural change, big activity change)? Docking/scoring functions notoriously miss these. Naming them in advance is the signal.

This is where your electronic-structure background contributes: reasoning about H-bond strength, desolvation, and electronic effects on hinge binding that generic scoring functions handle poorly.

## Next: notebook 02 — activity-cliff & prediction analysis

With `tyk2_chembl_sar.csv` in hand, notebook 02 will: compute pairwise similarity, identify activity cliffs quantitatively (SALI), and test whether simple structure-based scores track measured potency — quantifying the failure modes you predicted above.

This reuses machinery you already know from IC50Forge/KinaseSeek — but here it's in service of *target-specific reasoning*, not a standalone tutorial, which is the difference that matters for interviews.